In [1]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent          # notebooks/ → корень проекта
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import re
from collections import Counter
from src.db import DATA_RAW

df = pd.read_parquet(DATA_RAW / "cars.parquet")
print(df.shape)
texts = df["seller_description"].fillna("").str.lower()

# топ слов
words = Counter()
for t in texts.sample(5000):    # 5000 случайных для скорости
    for w in re.findall(r"\b[а-яё]{4,}\b", t):
        words[w] += 1
print("Топ-50 слов:")
for w, n in words.most_common(50):
    print(f"{n:6d}  {w}")

# топ биграмм
bigrams = Counter()
for t in texts.sample(5000):
    ws = re.findall(r"\b[а-яё]{3,}\b", t)
    for i in range(len(ws) - 1):
        bigrams[f"{ws[i]} {ws[i+1]}"] += 1
print("\nТоп-30 биграмм:")
for b, n in bigrams.most_common(30):
    print(f"{n:6d}  {b}")

(30152, 63)
Топ-50 слов:
  9855  система
  7324  автомобиля
  6987  автомобиль
  4311  безопасности
  4068  сидений
  4059  пробегом
  3562  авто
  3220  контроль
  2954  зеркал
  2832  датчик
  2792  рольф
  2596  водителя
  2583  автомобилей
  2507  года
  2485  комплектация
  2434  сиденья
  2387  обогрев
  2384  вашего
  2362  подушки
  2344  руля
  2170  кредит
  2148  передних
  2030  более
  2017  подогрев
  1951  фары
  1947  автомобили
  1920  салона
  1918  пассажира
  1846  электропривод
  1830  помощи
  1770  заднего
  1762  передние
  1713  наличии
  1700  центральный
  1667  салон
  1641  круиз
  1629  задний
  1623  регулировка
  1618  условия
  1613  климат
  1537  гарантия
  1527  диски
  1516  состоянии
  1507  двигателя
  1488  есть
  1478  проверка
  1468  задние
  1448  задних
  1437  пробег
  1404  обмен

Топ-30 биграмм:
  2311  подушки безопасности
  1938  вашего автомобиля
  1782  передних сидений
  1585  круиз контроль
  1567  помощи при
  1556  климат контроль

In [2]:
private = df[df["seller_type"].isin(["Частное лицо", "Собственник"])]
dealer = df[~df["seller_type"].isin(["Частное лицо", "Собственник"])]

print(f"Частников: {len(private)}, дилеров: {len(dealer)}")
print(f"Средняя длина описания: частники {private['seller_description'].str.len().mean():.0f}, дилеры {dealer['seller_description'].str.len().mean():.0f}")

Частников: 14348, дилеров: 15804
Средняя длина описания: частники 651, дилеры 2002


In [3]:
from collections import Counter
import re

private_texts = private["seller_description"].fillna("").str.lower()
words = Counter()
for t in private_texts:
    for w in re.findall(r"\b[а-яё]{4,}\b", t):
        words[w] += 1

print("Топ-40 слов среди ЧАСТНИКОВ:")
for w, n in words.most_common(40):
    print(f"{n:6d}  {w}")

# и биграммы
bigrams = Counter()
for t in private_texts:
    ws = re.findall(r"\b[а-яё]{3,}\b", t)
    for i in range(len(ws) - 1):
        bigrams[f"{ws[i]} {ws[i+1]}"] += 1

print("\nТоп-30 биграмм среди ЧАСТНИКОВ:")
for b, n in bigrams.most_common(30):
    print(f"{n:6d}  {b}")

Топ-40 слов среди ЧАСТНИКОВ:
 10688  автомобиль
  7415  машина
  7294  состоянии
  6257  авто
  6247  есть
  4750  пробег
  4369  года
  4273  торг
  4119  салон
  3826  полностью
  3614  комплектация
  3548  замена
  3502  продаю
  3240  только
  3232  масло
  3229  сидений
  3183  система
  3079  отличном
  3001  контроль
  2862  двигатель
  2748  очень
  2672  состояние
  2602  резина
  2572  работает
  2481  диски
  2429  полный
  2421  комплект
  2296  хорошем
  2259  владелец
  2224  вложений
  2193  заднего
  2125  масла
  2117  автомобиля
  2083  коробка
  2009  время
  1988  подогрев
  1974  было
  1969  кузов
  1820  один
  1815  кузову

Топ-30 биграмм среди ЧАСТНИКОВ:
  2483  отличном состоянии
  1851  хорошем состоянии
  1457  заднего вида
  1348  круиз контроль
  1232  любые проверки
  1228  климат контроль
  1210  замена масла
  1172  камера заднего
  1168  без дтп
  1024  официального дилера
  1022  два комплекта
  1014  полный привод
   993  идеальном состоянии
   979  

In [4]:
import re
import pandas as pd
import numpy as np
from src.db import DATA_RAW

df = pd.read_parquet(DATA_RAW / "cars.parquet")
text = df["seller_description"].fillna("").str.lower()

KEYWORDS = {
    # состояние
    "kw_good_state":    r"(идеальн|отличн|хорош|техническ)\w*\s+состоян",
    "kw_no_dtp":        r"без\s+дтп|не\s+бит|не\s+крашен|не\s+бита",

    # история / обслуживание
    "kw_one_owner":     r"один\s+владел|1\s+владел|один\s+хозяин",
    "kw_dealer_serv":   r"официальн\w+\s+дилер",
    "kw_serviced":      r"полностью\s+обслужен|замена\s+масла|сервисн\w+\s+книж",

    # вложения
    "kw_no_invest":     r"вложений\s+не\s+треб|без\s+вложений|вложений\s+нет",
    "kw_needs_invest":  r"вложений\s+треб|требует\s+вложений",

    # комплектация / бонусы
    "kw_max_trim":      r"максимальн\w+\s+комплектац|полн\w+\s+комплектац",
    "kw_two_sets":      r"два\s+комплект|2\s+комплект|зимн\w+\s+резин",
    "kw_no_complaints": r"без\s+нарекан",

    # продавец
    "kw_any_check":     r"любы\w+\s+проверк",
    "kw_sel_poehal":    r"сел\s+(и\s+)?поехал",
}

for flag, pattern in KEYWORDS.items():
    df[flag] = text.str.contains(pattern, regex=True, na=False)

C:\Users\Андрей\AppData\Local\Temp\ipykernel_5136\3746764648.py:34: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df[flag] = text.str.contains(pattern, regex=True, na=False)
C:\Users\Андрей\AppData\Local\Temp\ipykernel_5136\3746764648.py:34: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df[flag] = text.str.contains(pattern, regex=True, na=False)


In [5]:
rows = []
for flag in KEYWORDS:
    n = int(df[flag].sum())
    if n < 50:
        rows.append({"feature": flag, "n_true": n, "lift": None})
        continue
    m_t = df.loc[df[flag], "price"].median()
    m_f = df.loc[~df[flag], "price"].median()
    rows.append({
        "feature": flag,
        "n_true": n,
        "median_true": m_t,
        "median_false": m_f,
        "lift": round(m_t / m_f, 3),
    })

imp = pd.DataFrame(rows).sort_values("lift", ascending=False)
print(imp.to_string(index=False))

         feature  n_true  lift  median_true  median_false
       kw_no_dtp    2172 1.097    2300000.0     2097500.0
    kw_one_owner    3390 0.980    2057000.0     2100000.0
 kw_needs_invest     327 0.929    1950000.0     2100000.0
     kw_two_sets    1925 0.897    1900000.0     2119000.0
     kw_serviced    2148 0.838    1800000.0     2148000.0
    kw_no_invest    1244 0.775    1650000.0     2130000.0
kw_no_complaints     770 0.685    1459500.0     2130000.0
   kw_sel_poehal     865 0.605    1300000.0     2148000.0
   kw_good_state       0   NaN          NaN           NaN
  kw_dealer_serv       0   NaN          NaN           NaN
     kw_max_trim       0   NaN          NaN           NaN
    kw_any_check       1   NaN          NaN           NaN


In [6]:
# доля частников среди тех, у кого kw_no_invest = True
print(df.groupby("kw_no_invest")["seller_type"].value_counts(normalize=True).head(10))

# средняя цена по brand среди True и False
print(df[df["kw_no_invest"]].groupby("brand")["price"].median().sort_values().head(10))
print(df[~df["kw_no_invest"]].groupby("brand")["price"].median().sort_values().head(10))

kw_no_invest  seller_type                                    
False         Частное лицо                                       0.333264
              Собственник                                        0.121143
              Major Expert Новая Рига / Автомобили с пробегом    0.024422
              Fresh Москва                                       0.017089
              РОЛЬФ Северо-Запад | Автомобили с пробегом         0.011173
              PELETON                                            0.010343
              РОЛЬФ Юго-Восток                                   0.009893
              РОЛЬФ Алтуфьево                                    0.009167
              Дилерский центр Орехово                            0.008579
              РОЛЬФ Восток                                       0.007541
Name: proportion, dtype: float64
brand
Daewoo        150277.5
ЗАЗ           240000.0
Datsun        420000.0
Лада          487500.0
Peugeot       530000.0
Great Wall    550000.0
Fiat          620000.

In [7]:
sample = df["seller_description"].dropna().iloc[0]
print(repr(sample[:200]))

# почему не матчит?
import re
print(re.findall(r"(идеальн|отличн|хорош|техническ)\w*\s+состоян",
                 sample.lower()))

'Автомобиль в отличном состоянии. Машина находилась в бережной эксплуатации, владелец девушка.\nСостояние кузова норм, есть небольшие царапины, в машине не курили. Авто без вложений сел - поехал. Два ко'
['отличн']


In [8]:
for brand in ["Лада", "Kia", "Hyundai", "Renault", "Toyota", "BMW"]:
    sub = df[df["brand"] == brand]
    if sub["kw_no_invest"].sum() < 30:
        print(f"{brand}: мало данных, skip")
        continue
    t = sub.loc[sub["kw_no_invest"], "price"].median()
    f = sub.loc[~sub["kw_no_invest"], "price"].median()
    print(f"{brand:10s}  n_true={sub['kw_no_invest'].sum():4d}  "
          f"median_T={t:>10,.0f}  median_F={f:>10,.0f}  lift={t/f:.3f}")

Лада        n_true=  44  median_T=   487,500  median_F=   635,000  lift=0.768
Kia         n_true=  93  median_T= 1,350,000  median_F= 1,550,000  lift=0.871
Hyundai     n_true=  64  median_T= 1,325,000  median_F= 1,525,000  lift=0.869
Renault: мало данных, skip
Toyota      n_true=  98  median_T= 2,424,500  median_F= 2,524,000  lift=0.961
BMW         n_true= 116  median_T= 2,850,000  median_F= 4,200,000  lift=0.679


In [9]:
brands_to_check = ["Лада", "Kia", "Hyundai", "Toyota", "BMW", "Mercedes-Benz",
                   "Volkswagen", "Nissan", "Skoda", "Ford"]

for flag in ["kw_no_dtp", "kw_one_owner", "kw_needs_invest",
             "kw_two_sets", "kw_serviced", "kw_no_complaints", "kw_sel_poehal"]:
    print(f"\n=== {flag} ===")
    for brand in brands_to_check:
        sub = df[df["brand"] == brand]
        if sub[flag].sum() < 30:
            continue
        t = sub.loc[sub[flag], "price"].median()
        f = sub.loc[~sub[flag], "price"].median()
        print(f"  {brand:15s} n={sub[flag].sum():4d}  "
              f"T={t:>10,.0f}  F={f:>10,.0f}  lift={t/f:.3f}")


=== kw_no_dtp ===
  Лада            n=  81  T=   800,000  F=   610,000  lift=1.311
  Kia             n= 164  T= 1,840,000  F= 1,500,000  lift=1.227
  Hyundai         n= 107  T= 1,800,000  F= 1,500,000  lift=1.200
  Toyota          n= 143  T= 2,800,000  F= 2,498,500  lift=1.121
  BMW             n= 230  T= 4,225,000  F= 4,050,000  lift=1.043
  Mercedes-Benz   n= 251  T= 4,750,000  F= 4,000,000  lift=1.188
  Volkswagen      n=  87  T= 1,850,000  F= 1,527,000  lift=1.212
  Nissan          n=  65  T= 1,400,000  F= 1,266,000  lift=1.106
  Skoda           n=  63  T= 1,530,000  F= 1,300,000  lift=1.177
  Ford            n=  47  T= 1,080,000  F= 1,000,000  lift=1.080

=== kw_one_owner ===
  Лада            n= 106  T=   817,000  F=   600,000  lift=1.362
  Kia             n= 285  T= 1,600,000  F= 1,520,000  lift=1.053
  Hyundai         n= 215  T= 1,590,000  F= 1,500,000  lift=1.060
  Toyota          n= 172  T= 2,637,000  F= 2,499,999  lift=1.055
  BMW             n= 250  T= 5,622,500  F= 3,950,

In [12]:
sample = df["seller_description"].dropna().iloc[0]
print(repr(sample[:300]))
import re
print(re.findall(r"официальн\w+\s+дилер", sample.lower()))

'Автомобиль в отличном состоянии. Машина находилась в бережной эксплуатации, владелец девушка.\nСостояние кузова норм, есть небольшие царапины, в машине не курили. Авто без вложений сел - поехал. Два комплекта резины на литье. Подогрев передних сидений.\nЗамена масла и расходников производилась в мае 2'
[]


In [13]:
import re

sample = 'Автомобиль в отличном состоянии. Машина находилась в бережной эксплуатации, владелец девушка. Состояние кузова норм, есть небольшие царапины, в машине не курили. Авто без вложений сел - поехал. Два комплекта резины на литье.'

tests = {
    "kw_good_state":  r"(идеальн|отличн|хорош|техническ)[а-яё]*\s+состоян",
    "kw_sel_poehal":  r"сел\s+(и\s+)?поехал",
    "kw_two_sets":    r"два\s+комплект",
    "kw_no_invest":   r"без\s+вложений",
    "kw_any_check":   r"любы[а-яё]*\s+проверк",
}
for name, pat in tests.items():
    print(name, re.findall(pat, sample.lower()))

kw_good_state ['отличн']
kw_sel_poehal []
kw_two_sets ['два комплект']
kw_no_invest ['без вложений']
kw_any_check []


In [14]:
text = df["seller_description"].fillna("").str.lower()

for name, pat in tests.items():
    n = text.str.contains(pat, regex=True, na=False).sum()
    print(f"{name}: {n}")

C:\Users\Андрей\AppData\Local\Temp\ipykernel_5136\2199406608.py:4: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  n = text.str.contains(pat, regex=True, na=False).sum()


kw_good_state: 12515
kw_sel_poehal: 865
kw_two_sets: 1010
kw_no_invest: 305
kw_any_check: 1438
